In [67]:
# This will borrow heavily from the prior scraping script

import pandas as pd
import urllib.request
import urllib.error
import time
from bs4 import BeautifulSoup, Comment

In [68]:
headers = {'User-Agent':'Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36'}

In [69]:
def scrape_season_data(url):
    """
    Scrapes game score data for regular season games for a single NHL team and a single season.

    Args: url (str) . This is the gamelog page for one team's games for one season

    Returns: pd.DataFrame: a DataFrame containing game-level data on goals for and against, with ENGs removed.
    """

    req = urllib.request.Request(url, headers=headers)
    # Error handling
    try:
        #html = urlopen(url)
        with urllib.request.urlopen(req) as response:
            html = response.read().decode('utf-8')
    except urllib.error.HTTPError as e:
        print(f"HTTP error {e} - could not retrieve data from {url}") # return null, break, or something else
        return pd.DataFrame()
    except urllib.error.URLError as e:
        print(f"URL error {e}. The server could not be found for {url}.")
        return pd.DataFrame()
    
    # the program continues. If I return or break in the exception catch, "else" isn't needed.

    soup = BeautifulSoup(html, 'html.parser')

    # Get the season from the url

    parts = url.split('/')
    season = parts[-1][4:8]

    game_data = []

    #regular_season_table = None
    
    regular_season_table = soup.find('table', {'id': 'games'}) 

    #if not regular_season_table:
    #    comments = soup.find_all(string=lambda text: isinstance(text, Comment))

    #    for comment in comments:
    #        hidden_soup = BeautifulSoup(comment, 'html.parser')
    #        regular_season_table = hidden_soup.find('table', {'id': 'games'})
    #        if regular_season_table:
    #            break
    
    if not regular_season_table:
        print(f'Could not find the regular season table for the {season} season.')
        return pd.DataFrame()

    # Iterate through each row in the table

    for row in regular_season_table.tbody.find_all('tr'):

        time.sleep(3)
        
        # Skip rows without data or that are headers

        if 'class' in row.attrs and 'thead' in row['class']:
            continue

        # Try getting box score link here
        box = row.find('th')
        if not box:
            continue

        cols = row.find_all('td')
        #if len(cols) < 10:
        #    continue

        #print(cols[0])
        #print(cols[1])
        #print(cols[2])
        #print(cols[3])
        #print(cols[4])
        #print(cols[5])
        
        try:
            # Extract data from the game log table
            vis_gf = int(cols[2].text.strip())
            home_gf = int(cols[4].text.strip())
            #w_l = cols[5].text.strip() There is no W/L column in these pages
            ot_so = cols[5].text.strip()

            # Get box score url from column 1
            box_score_link = box.find('a')['href']
            box_score_url = f'https://www.hockey-reference.com{box_score_link}'
            #print(f'Checking the box score at {box_score_url} for ENGs.')

            # Get the team codes for both teams
            vis_team_link = cols[1].find('a')['href']
            vis_team_parts = vis_team_link.split('/')
            vis_team_code = vis_team_parts[-2]
            
            home_team_link = cols[3].find('a')['href']
            home_team_parts = home_team_link.split('/')
            home_team_code = home_team_parts[-2]

            # Adjust goal totals for shootouts
            if vis_gf > home_gf and ot_so == "SO":
                vis_gf -= 1
            elif vis_gf < home_gf and ot_so == "SO":
                home_gf -= 1

            # Scrape the box scores for ENGs

            vis_eng = 0
            home_eng = 0

            req_bs = urllib.request.Request(box_score_url, headers=headers)

            time.sleep(3)
                
            try:
                with urllib.request.urlopen(req_bs) as response_bs:
                    box_score_html = response_bs.read().decode('utf-8')
        
                box_score_soup = BeautifulSoup(box_score_html, 'html.parser')
            
                # Find the Scoring Summary table

                scoring_summary_table = box_score_soup.find('table', {'id': 'scoring'})

                if scoring_summary_table:
                    for scoring_row in scoring_summary_table.find_all('tr'):
                        
                        if 'class' in scoring_row.attrs and 'thead' in scoring_row['class']:
                            continue
                            
                        scoring_cols = scoring_row.find_all('td')
                        if len(scoring_cols) >= 3:
                            team_code = scoring_cols[1].text.strip()
                            goal_type = scoring_cols[2].text.strip()

                            # Check for "EN" in goal_type
                            if "EN" in goal_type:
                                if team_code == vis_team_code:
                                    vis_eng += 1
                                else:
                                    home_eng += 1
            except urllib.error.HTTPError as e:
                print(f"HTTP Error as {e} - Could not retrieve box score from {box_score_url}.")
            except urllib.error.URLError as e:
                print(f"URL Error as {e} - The server could not be found for {box_score_url}.")

            vis_gf -= vis_eng
            home_gf -= home_eng

            game_data.append({
                "SEASON": season, 
                "VISITOR_GF": vis_gf,
                "HOME_GF": home_gf
                })
        
        except(ValueError, IndexError) as e:
            print(f'Skipping a row due to a data error: {e}')
            continue
    
    return pd.DataFrame(game_data)
    

In [99]:
# Define the urls to scrape
# We can easily add to this list

urls_to_scrape = [
    "https://www.hockey-reference.com/leagues/NHL_2025_games.html",
]

# And an empty list to store data frames

all_data_frames = []


In [100]:
# Loop through urls and scrape data from each one

for url in urls_to_scrape:
    print(f"Scraping data from {url} ...")
    df = scrape_season_data(url)
    if not df.empty:
        all_data_frames.append(df)
    print(f"Finished scraping data from {url}.")

if all_data_frames:
    final_df = pd.concat(all_data_frames, ignore_index=True)

Scraping data from https://www.hockey-reference.com/leagues/NHL_2025_games.html ...
Finished scraping data from https://www.hockey-reference.com/leagues/NHL_2025_games.html.


In [101]:
final_df.head

<bound method NDFrame.head of      SEASON  VISITOR_GF  HOME_GF
0      2025           3        1
1      2025           1        3
2      2025           4        6
3      2025           3        2
4      2025           2        4
...     ...         ...      ...
1307   2025           5        1
1308   2025           0        4
1309   2025           5        7
1310   2025           2        5
1311   2025           3        4

[1312 rows x 3 columns]>

In [102]:
final_df.to_csv('scraping_homeice_20242025.csv', index=False)

In [80]:
df_2020 = final_df[final_df['SEASON'] == '2020']

In [81]:
df_2020.head

<bound method NDFrame.head of      SEASON  VISITOR_GF  HOME_GF
2542   2020           2        3
2543   2020           3        2
2544   2020           3        5
2545   2020           1        4
2546   2020           1        2
...     ...         ...      ...
3619   2020           2        1
3620   2020           2        6
3621   2020           2        3
3622   2020           3        2
3623   2020           2        3

[1082 rows x 3 columns]>

In [82]:
df_2020.to_csv('scraping_homeice_20192020.csv', index=False)